# 00_config
Shared configuration for the longitudinal recourse-audit project on KHPS 2019-2024.
Defines paths, global plotting rules (greyscale seaborn, dpi 600, png+pdf, no captions),
KHPS recoding helpers, and the actionable / target / covariate variable registry.
Run this notebook first, or `%run 00_config.ipynb` from other notebooks.

In [1]:
# --- Core imports ---
import os, warnings, itertools
import numpy as np
import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

# --- Project paths (resolve relative to notebooks/) ---
NB_DIR   = os.getcwd()
PROJ_DIR = os.path.abspath(os.path.join(NB_DIR, ".."))
DATA_DIR = os.path.join(PROJ_DIR, "data")
RES_DIR  = os.path.join(PROJ_DIR, "results")
FIG_DIR  = os.path.join(RES_DIR, "figures")
TAB_DIR  = os.path.join(RES_DIR, "tables")
for d in (FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)
print("PROJ_DIR:", PROJ_DIR)

PROJ_DIR: /home/claude/recourse_khp


In [2]:
# --- Global plotting rules: greyscale, dpi 600, png+pdf, no captions ---
sns.set_theme(style="whitegrid", context="paper")
sns.set_palette(sns.color_palette(["#111111","#444444","#777777","#a5a5a5","#cccccc"]))
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600,
    "font.family": "DejaVu Sans", "axes.grid": True,
    "grid.color": "#dddddd", "grid.linewidth": 0.6,
    "axes.edgecolor": "#333333", "image.cmap": "Greys",
})

def savefig(fig, name, tight=True):
    """Save a figure as both PNG and PDF at dpi 600 into results/figures. No caption."""
    if tight: fig.tight_layout()
    for ext in ("png","pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=600, bbox_inches="tight")
    print("saved:", name+".png /", name+".pdf")

def savetable(df, name, index=True):
    """Save a table as CSV into results/tables."""
    p=os.path.join(TAB_DIR, f"{name}.csv")
    df.to_csv(p, index=index, encoding="utf-8-sig")
    print("saved:", os.path.basename(p))
    return df

In [3]:
# --- KHPS wave registry (2nd panel: a=2019 ... f=2024) ---
WAVES = {"a":2019,"b":2020,"c":2021,"d":2022,"e":2023,"f":2024}
WAVE_SEQ = list(WAVES.keys())
WAVE_PAIRS = list(zip(WAVE_SEQ[:-1], WAVE_SEQ[1:]))   # 1-year (t, t+1)
WAVE_PAIRS_2Y = list(zip(WAVE_SEQ[:-2], WAVE_SEQ[2:])) # 2-year (t, t+2)
print("1y pairs:", [(WAVES[a],WAVES[b]) for a,b in WAVE_PAIRS])
print("2y pairs:", [(WAVES[a],WAVES[b]) for a,b in WAVE_PAIRS_2Y])

1y pairs: [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024)]
2y pairs: [(2019, 2021), (2020, 2022), (2021, 2023), (2022, 2024)]


In [4]:
# --- Variable registry ---
# Linkage
KEY = "PIDWON"                    # person id, stable across waves
HHKEY = "HHID"
# Actionable features (recourse may prescribe these) -- smoking excluded by design
ACTIONABLE = {
    "BMI"      : ["HT","WT"],            # BMI = WT/(HT/100)**2
    "ALC_FREQ" : ["D1"],                 # drinking frequency
    "ALC_AMT"  : ["D6"],                 # usual amount
    "ALC_BINGE": ["D7"],                 # binge frequency
    "PA_REG"   : ["P1"],                 # regular exercise (1/2)
    "PA_WALK"  : ["P2"],                 # walking days last week
}
# Non-actionable covariates (enter model, never prescribed)
COVARIATES = ["SEX","BIRTH_Y","EDU","H_INC_TOT","REGION1","DISA_YN"]
# Targets: chronic-disease onset. Primary = HTN; sensitivity = DM, DYS
TARGETS = {"HTN":("CD1_HTN","CD2_HTN"), "DM":("CD1_DM","CD2_DM"), "DYS":("CD1_DYS","CD2_DYS")}
PRIMARY_TARGET = "HTN"
# Actuarial / adverse-selection proxies (axis 3)
ACTUARIAL = ["ERGUN","INGUN","OUGUN","EROOP","INOOP","OUOOP_1","IN1YEAR",
             "I_PHI1_YN","I_FFS_YN"]
# Equity strata (axis 5)
EQUITY = ["SEX","AGEG","REGION1","EDU","INCQ"]
UNMET = "AH1"                     # unmet care experience (external validity)
print("registry loaded")

registry loaded


In [5]:
# --- Recoding helpers ---
MISSING_CODES = {-9,-8,-1,9,99,999}   # KHPS non-response / not-applicable sentinels

def clean_missing(s, extra=None):
    """Map KHPS sentinel codes to NaN."""
    codes=set(MISSING_CODES) | (set(extra) if extra else set())
    return s.where(~s.isin(codes))

def yn_disease(s):
    """CD1_*: 1=has disease ->1, 2=no ->0, else NaN."""
    return s.map({1:1, 2:0})

def compute_bmi(df, ht="HT", wt="WT"):
    ht_m = clean_missing(df[ht])/100.0
    wt_k = clean_missing(df[wt])
    bmi  = wt_k/(ht_m**2)
    return bmi.where(bmi.between(12,60))   # plausibility filter

def age_group(age):
    bins=[18,29,39,49,59,69,200]
    labs=["19-29","30-39","40-49","50-59","60-69","70+"]
    return pd.cut(age, bins=bins, labels=labs)

def income_quintile(s):
    x=clean_missing(s)
    try:
        return pd.qcut(x, 5, labels=["Q1","Q2","Q3","Q4","Q5"], duplicates="drop")
    except Exception:
        return pd.Series(index=s.index, dtype="object")
print("helpers loaded")

helpers loaded


In [6]:
# --- Loader ---
def load_ind(wave, usecols=None):
    """Load one wave of the IND (individual) DB, add year/age."""
    path=os.path.join(DATA_DIR, f"{wave}_ind.sas7bdat")
    df,_=pyreadstat.read_sas7bdat(path, usecols=usecols)
    df["wave"]=wave; df["year"]=WAVES[wave]
    if "BIRTH_Y" in df: df["age"]=df["year"]-df["BIRTH_Y"]
    return df

def load_hh(wave, usecols=None):
    path=os.path.join(DATA_DIR, f"{wave}_hh.sas7bdat")
    df,_=pyreadstat.read_sas7bdat(path, usecols=usecols)
    df["wave"]=wave; df["year"]=WAVES[wave]
    return df

print("00_config ready")

00_config ready
